# ARC: tiny data loader and symbolic verifier

This notebook accompanies [Bayless et al., arXiv:2511.09008v1](https://arxiv.org/abs/2511.09008v1). It loads the public **ConditionalQA** source dataset and demonstrates only the deterministic SMT-verification core of ARC. The paper's derived **CONDITIONALQA-LOGIC** data and full ARC implementation were not publicly linked as of 2026-11-08.

In [ ]:
!pip -q install z3-solver

import json
import urllib.request
from pathlib import Path
from pprint import pprint

from z3 import And, Bool, BoolVal, Int, Not, Or, Solver, unsat

## 1. Load the public source dataset

The official ConditionalQA repository publishes the train/dev/test files under CC BY-SA 4.0. The newer paper transforms a subset into five logical labels, but does not publish those transformed rows.

In [ ]:
DEV_URL = 'https://raw.githubusercontent.com/haitian-sun/ConditionalQA/master/v1_0/dev.json'
with urllib.request.urlopen(DEV_URL) as response:
    conditionalqa_dev = json.load(response)

print(f'Loaded {len(conditionalqa_dev):,} ConditionalQA development examples.')
for row in conditionalqa_dev[:2]:
    pprint({key: row[key] for key in ('id', 'scenario', 'question', 'answers', 'not_answerable')})
    print()

## 2. Inspect the reconstructed JSON format

These are synthetic examples shaped from the fields described in the paper. They are included because the actual CONDITIONALQA-LOGIC records were not released.

In [ ]:
examples_path = Path('examples.json')
if examples_path.exists():
    logic_examples = json.loads(examples_path.read_text())
else:
    logic_examples = [
        {
            'id': 'synthetic-valid-001',
            'policy_text': 'A passenger is eligible for a refund if the flight is cancelled or the departure is delayed by at least five hours.',
            'scenario': "The passenger's flight is delayed by six hours and is not cancelled.",
            'question': 'Is the passenger eligible for a refund?',
            'answer': 'Yes.',
            'conditions': ['The departure is delayed by at least five hours.'],
            'premises_smt2': '(and (= delay_hours 6) (= flight_cancelled false))',
            'conclusion_smt2': 'refund_eligible',
            'label': 'Valid',
            'transformation': 'none',
            'is_released_paper_example': False,
        },
        {
            'id': 'synthetic-satisfiable-001',
            'policy_text': 'A passenger is eligible for a refund if the flight is cancelled or the departure is delayed by at least five hours.',
            'scenario': 'The departure delay is unknown and the flight is not cancelled.',
            'question': 'Is the passenger eligible for a refund?',
            'answer': 'Yes.',
            'conditions': [],
            'premises_smt2': '(= flight_cancelled false)',
            'conclusion_smt2': 'refund_eligible',
            'label': 'Satisfiable',
            'transformation': 'necessary delay condition removed',
            'is_released_paper_example': False,
        },
    ]

pprint(logic_examples)

## 3. Minimal symbolic answer verification

ARC normally asks LLMs to produce the formulas. Here we write them manually, then classify a premise/conclusion pair using the paper's four solver-grounded outcomes.

In [ ]:
delay_hours = Int('delay_hours')
flight_cancelled = Bool('flight_cancelled')
refund_eligible = Bool('refund_eligible')

# A compact formal policy model. The domain constraint prevents negative delays.
policy = And(
    delay_hours >= 0,
    refund_eligible == Or(flight_cancelled, delay_hours >= 5),
)

def is_unsat(*formulas):
    solver = Solver()
    solver.add(*formulas)
    return solver.check() == unsat

def verify(premises, conclusion):
    if is_unsat(policy, premises):
        return 'Impossible'
    if is_unsat(policy, premises, conclusion):
        return 'Invalid'
    if is_unsat(policy, premises, Not(conclusion)):
        return 'Valid'
    return 'Satisfiable'

tests = {
    'six-hour delay implies eligible': (And(delay_hours == 6, Not(flight_cancelled)), refund_eligible),
    'two-hour delay implies eligible': (And(delay_hours == 2, Not(flight_cancelled)), refund_eligible),
    'unknown delay may be eligible': (Not(flight_cancelled), refund_eligible),
    'negative delay premise': (delay_hours == -1, refund_eligible),
}

for name, (premises, conclusion) in tests.items():
    print(f'{name:38s} -> {verify(premises, conclusion)}')

### Scope

This is an educational reduction, not a reproduction of the reported results. A faithful reproduction would additionally need the unreleased transformed dataset/policy models, LLM translation prompts and models, redundant translations, logical-equivalence aggregation, human-vetting workflow, and feedback generation.